In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import nibabel as nib

In [3]:
# DEFs

# json loader check
def load_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return {}

# NIfTI info
def get_nifti_info(path):
    img = nib.load(str(path))
    hdr = img.header
    return img.shape, hdr.get_zooms()


In [4]:
# DIRs

WORK_NAME = "M01_ds003768"


DATA_ROOT = Path("/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768")
RESULTS_ROOT = DATA_ROOT.parent / "rsCVR_Results"
WORK_ROOT = RESULTS_ROOT / WORK_NAME

INDEX_DIR = WORK_ROOT / "index"
LOG_DIR = WORK_ROOT / "logs"
QC_DIR = WORK_ROOT / "qc"
RAPIDTIDE_DIR = WORK_ROOT / "rapidtide"
SPM_DIR = WORK_ROOT / "spm_preproc"
SUMMARY_DIR = WORK_ROOT / "summary"
MANIFEST_DIR = WORK_ROOT / "manifests"

for d in [RESULTS_ROOT, WORK_ROOT, INDEX_DIR, LOG_DIR, QC_DIR, RAPIDTIDE_DIR, SPM_DIR, SUMMARY_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT    :", DATA_ROOT)
print("RESULTS_ROOT :", RESULTS_ROOT)
print("WORK_NAME    :", WORK_NAME)
print("WORK_ROOT    :", WORK_ROOT)

DATA_ROOT    : /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768
RESULTS_ROOT : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results
WORK_NAME    : M01_ds003768
WORK_ROOT    : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768


In [5]:
# DIRs for Subjects
subject_dirs = sorted([
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and p.name.lower().startswith("sub-")
])

print(f"Found {len(subject_dirs)} subject folders")
print([p.name for p in subject_dirs])

Found 33 subject folders
['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-28', 'sub-29', 'sub-30', 'sub-31', 'sub-32', 'sub-33']


In [6]:
# Subject disc

def build_subject_record(sub_dir: Path):
    sub = sub_dir.name
    anat_dir = sub_dir / "anat"
    func_dir = sub_dir / "func"

    t1_nii  = anat_dir / f"{sub}_T1w.nii.gz"
    t1_json = anat_dir / f"{sub}_T1w.json"

    r1_nii  = func_dir / f"{sub}_task-rest_run-1_bold.nii.gz"
    r1_json = func_dir / f"{sub}_task-rest_run-1_bold.json"

    r2_nii  = func_dir / f"{sub}_task-rest_run-2_bold.nii.gz"
    r2_json = func_dir / f"{sub}_task-rest_run-2_bold.json"

    rec = {
        "subject": sub,
        "subject_dir": str(sub_dir),

        "anat_dir_exists": anat_dir.exists(),
        "func_dir_exists": func_dir.exists(),

        "t1_nii": str(t1_nii),
        "t1_json": str(t1_json),
        "rest1_nii": str(r1_nii),
        "rest1_json": str(r1_json),
        "rest2_nii": str(r2_nii),
        "rest2_json": str(r2_json),

        "has_t1_nii": t1_nii.exists(),
        "has_t1_json": t1_json.exists(),
        "has_rest1_nii": r1_nii.exists(),
        "has_rest1_json": r1_json.exists(),
        "has_rest2_nii": r2_nii.exists(),
        "has_rest2_json": r2_json.exists(),
    }

    rec["complete_for_analysis"] = all([
        rec["anat_dir_exists"],
        rec["func_dir_exists"],
        rec["has_t1_nii"],
        rec["has_t1_json"],
        rec["has_rest1_nii"],
        rec["has_rest1_json"],
        rec["has_rest2_nii"],
        rec["has_rest2_json"],
    ])

    return rec

records = [build_subject_record(sub_dir) for sub_dir in subject_dirs]
df_index = pd.DataFrame(records)

print(df_index.shape)
df_index

(33, 17)


,subject,subject_dir,anat_dir_exists,func_dir_exists,t1_nii,t1_json,rest1_nii,rest1_json,rest2_nii,rest2_json,has_t1_nii,has_t1_json,has_rest1_nii,has_rest1_json,has_rest2_nii,has_rest2_json,complete_for_analysis
0,sub-01,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
1,sub-02,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
2,sub-03,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
3,sub-04,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
4,sub-05,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
5,sub-06,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
6,sub-07,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
7,sub-08,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
8,sub-09,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,True,True,True,True,True
9,sub-10,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,True,True,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenN

In [7]:
df_ok = df_index[df_index["complete_for_analysis"]].copy()
df_missing = df_index[~df_index["complete_for_analysis"]].copy()

print(f"Complete subjects: {len(df_ok)}")
print(f"Missing subjects : {len(df_missing)}")

if len(df_missing) > 0:
    display(df_missing[[
        "subject",
        "has_t1_nii", "has_t1_json",
        "has_rest1_nii", "has_rest1_json",
        "has_rest2_nii", "has_rest2_json"
    ]])
else:
    print("All subjects are complete for analysis.")

Complete subjects: 33
Missing subjects : 0
All subjects are complete for analysis.


In [8]:
# Read metadata from JSON files

meta_rows = []
for _, row in df_ok.iterrows():
    j1 = load_json_safe(row["rest1_json"])
    j2 = load_json_safe(row["rest2_json"])
    jt = load_json_safe(row["t1_json"])
    meta_rows.append({
        "subject": row["subject"],
        "rest1_TR": j1.get("RepetitionTime", np.nan),
        "rest2_TR": j2.get("RepetitionTime", np.nan),
        "rest1_TaskName": j1.get("TaskName", ""),
        "rest2_TaskName": j2.get("TaskName", ""),
        "t1_Modality": jt.get("Modality", ""),
        "t1_MRAcquisitionType": jt.get("MRAcquisitionType", ""),
    })
df_meta = pd.DataFrame(meta_rows)

# # TR consistency check
# print("Unique TR in run-1:", sorted(df_meta["rest1_TR"].dropna().unique().tolist()))
# print("Unique TR in run-2:", sorted(df_meta["rest2_TR"].dropna().unique().tolist()))
df_meta

,subject,rest1_TR,rest2_TR,rest1_TaskName,rest2_TaskName,t1_Modality,t1_MRAcquisitionType
0,sub-01,2.1,2.1,rest,rest,MR,3D
1,sub-02,2.1,2.1,rest,rest,MR,3D
2,sub-03,2.1,2.1,rest,rest,MR,3D
3,sub-04,2.1,2.1,rest,rest,MR,3D
4,sub-05,2.1,2.1,rest,rest,MR,3D
5,sub-06,2.1,2.1,rest,rest,MR,3D
6,sub-07,2.1,2.1,rest,rest,MR,3D
7,sub-08,2.1,2.1,rest,rest,MR,3D
8,sub-09,2.1,2.1,rest,rest,MR,3D
9,sub-10,2.1,2.1,rest,rest,MR,3D


In [9]:
# NIfTI :: Read shapes & voxel sizes

shape_rows = []
for _, row in df_ok.iterrows():
    try:
        t1_shape, t1_zooms = get_nifti_info(row["t1_nii"])
        r1_shape, r1_zooms = get_nifti_info(row["rest1_nii"])
        r2_shape, r2_zooms = get_nifti_info(row["rest2_nii"])
        shape_rows.append({
            "subject": row["subject"],
            "t1_shape": t1_shape,
            "t1_zooms": t1_zooms,
            "rest1_shape": r1_shape,
            "rest1_zooms": r1_zooms,
            "rest2_shape": r2_shape,
            "rest2_zooms": r2_zooms,
        })
    except Exception as e:
        shape_rows.append({
            "subject": row["subject"],
            "t1_shape": str(e),
            "t1_zooms": None,
            "rest1_shape": None,
            "rest1_zooms": None,
            "rest2_shape": None,
            "rest2_zooms": None,
        })
df_shapes = pd.DataFrame(shape_rows)

# Shape consistency check
print("Unique rest run-1 shapes:")
display(df_shapes["rest1_shape"].value_counts())
print("Unique rest run-2 shapes:")
display(df_shapes["rest2_shape"].value_counts())

df_shapes

Unique rest run-1 shapes:


rest1_shape
(80, 80, 35, 286)    33
Name: count, dtype: int64

Unique rest run-2 shapes:


rest2_shape
(80, 80, 35, 286)    33
Name: count, dtype: int64

,subject,t1_shape,t1_zooms,rest1_shape,rest1_zooms,rest2_shape,rest2_zooms
0,sub-01,"(192, 256, 256)","(0.9999971, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
1,sub-02,"(192, 256, 256)","(1.0, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
2,sub-03,"(192, 256, 256)","(0.99999654, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
3,sub-04,"(192, 256, 256)","(0.9999973, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
4,sub-05,"(192, 256, 256)","(1.0000013, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
5,sub-06,"(192, 256, 256)","(0.9999992, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
6,sub-07,"(192, 256, 256)","(1.000001, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 3.9999998, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
7,sub-08,"(192, 256, 256)","(1.0000037, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
8,sub-09,"(192, 256, 256)","(1.0000008, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"
9,sub-10,"(192, 256, 256)","(1.0000021, 1.0, 1.0)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)","(80, 80, 35, 286)","(3.0, 3.0, 4.0, 2.1)"


In [10]:
# Save as CSV files

index_csv = INDEX_DIR / "subject_index.csv"
df_index.to_csv(index_csv, index=False)

meta_csv = INDEX_DIR / "subject_meta.csv"
df_meta.to_csv(meta_csv, index=False)

shape_csv = INDEX_DIR / "subject_shapes.csv"
df_shapes.to_csv(shape_csv, index=False)


print(index_csv)
print(meta_csv)
print(shape_csv)

/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/index/subject_index.csv
/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/index/subject_meta.csv
/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/index/subject_shapes.csv


In [11]:
# Final check

print("Summary :: ")
print(f"Total subject folders : {len(subject_dirs)}")
print(f"Complete subjects     : {len(df_ok)}")
print(f"Missing subjects      : {len(df_missing)}")
print("Run-1 TR values       :", sorted(df_meta["rest1_TR"].dropna().unique().tolist()))
print("Run-2 TR values       :", sorted(df_meta["rest2_TR"].dropna().unique().tolist()))

Summary :: 
Total subject folders : 33
Complete subjects     : 33
Missing subjects      : 0
Run-1 TR values       : [2.1]
Run-2 TR values       : [2.1]
